In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, set_key
from datetime import datetime, timedelta
from time import sleep

# === Load environment and GitHub tokens ===
env_path = "All_Tokens.env"
load_dotenv(env_path)

# Load up to 6 tokens
tokens = []
for i in range(1, 7):
    token = os.getenv(f"GITHUB_TOKEN_{i}")
    if token:
        tokens.append(token)
    else:
        print(f"⚠️ GITHUB_TOKEN_{i} not found in All_Tokens.env")

if not tokens:
    raise ValueError("❌ No GitHub tokens found.")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    print(f"🔁 Using token GITHUB_TOKEN_{token_index + 1} of {len(tokens)}")
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === GitHub API Query ===
def run_query(created_range, lang, page):
    url = (
        f"https://api.github.com/search/repositories"
        f"?q=stars:>50+fork:false+archived:false+language:{lang}+created:{created_range}"
        f"&per_page=100&page={page}"
    )
    try:
        response = requests.get(url, headers=get_headers(), timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Request error: {e}")
        sleep(10)
        return None

# === Search configuration ===
output_dir = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "step1_search_output.csv")

initial_window_days = 7
min_window_days = 1 / 24  # 1 hour
max_window_days = 30
end_date = datetime(2024, 12, 31)
languages = ["Kotlin", "Java", "Dart"]

# === Main Loop ===
for lang in languages:
    env_key = f"START_DATE_{lang.upper()}"
    start_date_str = os.getenv(env_key, "2008-01-01")
    try:
        start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    except ValueError:
        raise ValueError(f"Invalid date format in .env for {env_key}. Use YYYY-MM-DD")

    print(f"▶️ Starting {lang} from {start_date.date()}")

    window_days = initial_window_days
    current = start_date

    while current < end_date:
        next_date = current + timedelta(days=window_days)
        if next_date > end_date:
            next_date = end_date

        created_range = f"{current.isoformat()}..{next_date.isoformat()}"
        total_fetched = 0
        page = 1
        items = []

        while page <= 10:
            data = run_query(created_range, lang, page)
            if data is None:
                print(f"⏸ Pausing due to error at {created_range} page {page}")
                break
            fetched = data.get("items", [])
            if not fetched:
                break
            items.extend(fetched)
            total_fetched += len(fetched)
            print(f"✅ {lang} | {created_range} | Page {page} | Fetched {len(fetched)}")
            if len(fetched) < 100:
                break
            page += 1
            sleep(1)

        # Save window results
        if items:
            df_window = pd.DataFrame([{
                "full_name": repo["full_name"],
                "html_url": repo["html_url"],
                "language": repo["language"],
                "created_at": repo["created_at"],
                "description": repo.get("description", ""),
                "topics": ",".join(repo.get("topics", [])),
                "name": repo.get("name", ""),
                "stars": repo.get("stargazers_count", 0),
            } for repo in items])
            if not os.path.exists(output_path):
                df_window.to_csv(output_path, index=False)
            else:
                df_window.to_csv(output_path, mode='a', header=False, index=False)

        # Update .env for this language
        set_key(env_path, env_key, str(next_date.date()))

        # Adaptive window adjustment
        if total_fetched >= 1000 and window_days > min_window_days:
            window_days = max(min_window_days, window_days / 2)
            print(f"⬇️ Shrinking window to {window_days:.4f} days")
        elif total_fetched < 300 and window_days < max_window_days:
            window_days = min(max_window_days, window_days * 2)
            print(f"⬆️ Expanding window to {window_days:.4f} days")
        else:
            current = next_date

    # ✅ Reset the START_DATE_<LANGUAGE> after completion
    set_key(env_path, env_key, "2008-01-01")
    print(f"🔄 Reset {env_key} in .env for future full runs.\n")

print("✅ Completed full adaptive search.")


▶️ Starting Kotlin from 2008-01-01
🔁 Using token GITHUB_TOKEN_1 of 6
⬆️ Expanding window to 14.0000 days
🔁 Using token GITHUB_TOKEN_2 of 6
⬆️ Expanding window to 28.0000 days
🔁 Using token GITHUB_TOKEN_3 of 6
⬆️ Expanding window to 30.0000 days
🔁 Using token GITHUB_TOKEN_4 of 6
🔁 Using token GITHUB_TOKEN_5 of 6
🔁 Using token GITHUB_TOKEN_6 of 6
🔁 Using token GITHUB_TOKEN_1 of 6
🔁 Using token GITHUB_TOKEN_2 of 6
🔁 Using token GITHUB_TOKEN_3 of 6


KeyboardInterrupt: 